In [0]:
%run ./cid_mapping_common_business

In [0]:
def cid_update(task_id):
    new_cid_df = (get_cidGroup_by_process(task_id)
        .filter(F.col("record_type") == F.lit(CID_MATCH_RECORD_TYPE_CM))
        .filter(F.coalesce(F.col("mapping_conusmer_id"), F.lit("")) != F.coalesce(F.col("new_mapping_conusmer_id"), F.lit("")))
        .select(
            F.col("task_id"), 
            F.col("mrkt_code"), 
            F.col("mapping_conusmer_id"), 
            F.col("new_mapping_conusmer_id")  
        )
        .distinct())
    
    # update t_transaction_master cid 
    base_table = DeltaTable.forName(spark, f"{get_env_config('golden_consumer_master_database')}.t_transaction_master")

    (base_table.alias("b")
        .merge(new_cid_df.alias("u"),  f""" b.tran_mrkt_code = u.mrkt_code and b.tran_mapping_consumer_id = u.mapping_conusmer_id """)
        .whenMatchedUpdate(
            set = {
                "tran_mapping_consumer_id": F.col("u.new_mapping_conusmer_id"),
                "task_id": F.col("u.task_id"),
            }
        )
        .execute())


def master_transaction_merge(task_id):
    cid_mapping_df = (get_cidGroup_by_process(task_id)
        .filter(F.col("record_type") == F.lit(CID_MATCH_RECORD_TYPE_BATCH))
        .select(
            "mrkt_code",
            "srcc_id",
            "new_mapping_conusmer_id"
        )
        .distinct())

    t_clean_consumer = (get_RL_clear_consumer_df(task_id)
        .withColumn("rank_num", 
                    F.row_number().over(
                        Window
                        .partitionBy(F.col("srcc_mrkt_code"), F.col("srcc_brnd_code"), F.col("srcc_srcs_code"), F.col("srcc_consumerid"))
                        .orderBy(F.col("srcc_sourcetimestamp").desc()))
        )
        .filter(F.col("rank_num") == 1)
        .select(
            F.expr("uuid()").alias("tran_id"),
            F.col("srcc_mrkt_code").alias("tran_mrkt_code"), 
            F.col("srcc_brnd_code").alias("tran_brnd_code"), 
            F.col("srcc_srcs_code").alias("tran_srcs_code"),
            F.col("srcc_consumerid").alias("tran_order_id"), 
            F.col("srcc_action").alias("tran_action"),
            F.col("srcc_id").alias("tran_srcc_id"),
            F.lit(task_id).alias("task_id"),
            F.current_timestamp().alias("creation_dt"),
            F.current_timestamp().alias("update_dt")
        ))
     
    
    batch_trans_df = (t_clean_consumer.alias("tcc")
        .join(cid_mapping_df.alias("cmd"), (F.col("tran_mrkt_code") == F.col("mrkt_code")) & (F.col("tran_srcc_id") == F.col("srcc_id")), "left")
        .select(
            F.col("tcc.*"),
            F.col("cmd.new_mapping_conusmer_id").alias("tran_mapping_consumer_id")
        )
    )
    
    # merge
    base_table = DeltaTable.forName(spark, f"{get_env_config('golden_consumer_master_database')}.t_transaction_master")

    (base_table.alias("b")
        .merge(batch_trans_df.alias("u"),  
               f""" b.tran_mrkt_code = u.tran_mrkt_code and b.tran_brnd_code = u.tran_brnd_code and b.tran_srcs_code = u.tran_srcs_code and b.tran_order_id = u.tran_order_id """
        )
        .whenMatchedUpdate(
            set = {
                "tran_mapping_consumer_id": F.col("u.tran_mapping_consumer_id"),
                "tran_action": F.col("u.tran_action"),
                "tran_srcc_id": F.col("u.tran_srcc_id"),
                "task_id": F.col("u.task_id"),
                "update_dt": F.col("u.update_dt"),
            }
        )
        .whenNotMatchedInsertAll()
        .execute())

In [0]:
def cid_merge_process(task_id):
    cid_update(task_id)
    master_transaction_merge(task_id)


In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")



with StepLogger("3.2_merge_trans_master", "03-2", "consumerlist", task_id=task_id) as logger:
    cid_merge_process(task_id)